In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
events_bronze = "/Volumes/main/lakehouse_marketing/bronze/events/"

df_events_bronze = spark.read\
                    .format("delta")\
                    .option("header", "true")\
                    .load(events_bronze)

display(df_events_bronze.limit(5))
display(df_events_bronze.printSchema())

event_id,user_id,campaign_id,event_type,event_timestamp,ingestion_timestamp,source_file
bdd640fb-0667-4ad1-9c80-317fa3b1799d,23b8c1e9-3924-46de-beb1-3b9046685257,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,view,2020-01-18T12:28:35.290095,2026-02-04T14:40:28.785Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv
0822e8f3-6c03-4199-972a-846916419f82,3b8faa18-37f8-488b-97fc-695a07a0ca6e,8fadc1a6-06cb-4fb3-9a1d-e644815ef6d1,view,1981-02-25T21:46:23.887436,2026-02-04T14:40:28.785Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv
6b65a6a4-8b81-48f6-b38a-088ca65ed389,47378190-96da-4dac-b2ff-5d2a386ecbe0,c241330b-01a9-471f-9e8a-774bcf36d58b,purchase,2015-03-16T02:48:25.304611,2026-02-04T14:40:28.785Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv
47229389-571a-4876-ac30-7511b2b9437a,null,c37459ee-f50b-4a63-b71e-cd7b27cd8130,click,1988-11-18T08:30:37.826879,2026-02-04T14:40:28.785Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv
5be6128e-18c2-4797-a142-ea7d17be3111,null,43b7a3a6-9a8d-4a03-980d-7b71d8f56413,view,2015-04-11T22:29:15.960319,2026-02-04T14:40:28.785Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv


root
 |-- event_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



* Selecionando as colunas

In [0]:
df = df_events_bronze.select(
                        "event_id",    
                        "user_id",
                        "campaign_id",
                        "event_type",
                        "event_timestamp",
                        "source_file",
                        "ingestion_timestamp"
)

##########################################################
# Aplicação / Reforço da Tipagem para (event_timestamp)
#########################################################
df_typed = df\
            .withColumn("event_timestamp", F.to_timestamp("event_timestamp"))


##########################################################################
#Nomalização para a coluna event_type (Precisa aplicar upper + trim)
##########################################################################
df_normalized = df_typed.withColumn("event_type", F.upper(F.trim(F.col("event_type"))))


display(df_normalized.select("event_type").distinct())
display(df_normalized.count())


event_type
CLICK
VIEW
PURCHASE


100000

#### Regras de Transformação

* Check de nulos

In [0]:
# Verificando a quantidade de nulos na base
df_normalized.groupBy(F.col("user_id").isNull().alias("is_null")).count().show()

+-------+-----+
|is_null|count|
+-------+-----+
|   true|10067|
|  false|89933|
+-------+-----+



In [0]:
# # # REGRA 1: Remover eventos sem `user_id`
# df_valid = df_normalized.filter(F.col("user_id").isNotNull())

# # REGRA 2: Trazer apenas o conjunto de eventos permitidos
# allowed_events = ['VIEW', 'PURCHASE', 'CLICK']
 
# df_valid = df_valid.filter(F.upper(F.col("event_type")).isin(allowed_events))

# # REGRA 3: Trazer apenas `event_timestamp` validos
# df_valid = df_valid.filter(F.col("event_timestamp").isNotNull())
# display(df_valid)

Vamos aplicar 3 regras:
* REGRA 1: Remover eventos sem `user_id`
* REGRA 2: Trazer apenas o conjunto de eventos permitidos
* REGRA 3: Trazer apenas `event_timestamp` validos



In [0]:
# Define os eventos permitos, conforme a documentação
allowed_events = ['VIEW', 'PURCHASE', 'CLICK']


df_with_rules = df_normalized.withColumn(
    "rejection_reason",
    F.when(F.col("user_id").isNull(), "NULL_USER_ID")
     .when(~F.col("event_type").isin(allowed_events), "INVALID_EVENT_TYPE")
     .when(F.col("event_timestamp").isNull(), "NULL_EVENT_TIMESTAMP")
     .otherwise(None)
)

display(df_with_rules.sample(0.001))

event_id,user_id,campaign_id,event_type,event_timestamp,source_file,ingestion_timestamp,rejection_reason
390fecf5-e841-4a39-b901-2b9c6057153a,d1366db1-5a57-4607-a7e5-46468430f0fa,caeb9959-7ca2-4582-9854-09d9de22dc3f,VIEW,1997-01-16T18:22:58.179Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
108e04ac-99c6-407f-ac81-4510c42eafc8,e900b66f-6083-4f88-97dd-a51ea4dc4d4e,b99b70d6-bc6a-48a7-be33-2380b7aa53b6,PURCHASE,1991-05-30T11:43:37.155Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
0f54c086-4505-41e1-8fc4-5e9eb60afa9c,a173e274-37fc-411b-826f-081a0651a897,72c7a202-062e-4778-9f0b-5135856638c8,VIEW,1992-08-18T08:27:38.845Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
2e89bdbd-5369-4c45-877e-0371ed140bfa,f48783eb-939f-4536-a7ba-a3dd2c2bd5a3,36cf28af-9d64-42ce-a8de-4479911eb1cd,VIEW,2011-07-26T22:47:08.638Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
5aa249bf-c827-4447-b8d8-e484d4e384a3,6c589836-3a04-44d7-a113-6ce039ba7d2f,599a20f1-81dc-457c-964c-af2f6d442c3f,VIEW,2024-07-10T18:48:22.471Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
cc1b5e2e-18f9-49a7-8551-4ed61c000740,aa6d7584-8846-49fb-b002-2232c5c04ac0,b23f9a7c-e7b3-4985-83b7-367ed999b615,VIEW,1989-03-21T13:54:38.067Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
c77a0940-b1cc-4f7c-a1c7-041e67cfffa0,236f8f92-d552-43ef-b1a1-91c7f810be73,9809c9e5-42fe-4589-9db0-0973203a604d,CLICK,1978-08-17T22:30:19.263Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
61f55560-0950-48b5-946d-b2006d4f2f52,6a216452-c62d-4e22-bb80-7727bb2a4f05,a8ea3f89-bd61-4466-8b33-d22dbf595974,VIEW,2014-05-03T19:46:24.148Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
49986f5a-1895-485b-9235-a4e141316886,3eec3946-a7c5-4775-a955-c00b8e7efbc3,a61b0097-973d-4f96-a0c7-5691d45958b6,PURCHASE,1972-08-17T00:40:06.513Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
79c6078d-b629-43fc-b801-ac19726c53f0,5885da4e-c0c0-4b60-9ef1-81ccc92d5726,ddd1c03d-303a-47dd-b8c0-9805a4f4219c,VIEW,2022-05-07T20:49:30.559Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null


* Separar válidos e rejeitados

In [0]:
df_valid = df_with_rules.filter(F.col("rejection_reason").isNull())
df_rejected = df_with_rules.filter(F.col("rejection_reason").isNotNull())

display(df_valid.sample(0.001))
display(df_rejected.sample(0.001))

event_id,user_id,campaign_id,event_type,event_timestamp,source_file,ingestion_timestamp,rejection_reason
df0ddef1-90c4-4e45-95cf-37bae98965f2,2980f6ff-266c-47fb-9cd8-1a8ba8d6faf6,c4e775b9-2dca-465a-a504-fd28de6ae459,VIEW,2009-12-10T20:43:51.594Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
2e3d9de2-e504-41c6-8312-153d484f7f7c,d83940c9-7e86-47f8-93cf-44fe17c03e28,353daedc-d69b-4eb1-a9a7-f5e05c8488db,PURCHASE,1991-12-03T22:52:48.785Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
b10ccf77-ba4d-4c8e-955c-8139384243b4,460f01a5-4480-402d-a067-902bf099e2b6,178b55e7-f406-46e0-b301-3b087b487db6,VIEW,1989-07-21T08:59:17.722Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
dd8d0747-42d3-41aa-9526-db8e922febcf,f3fce192-cf8e-41a3-93de-2da040ff5822,b72a6928-9c78-41de-bf84-9eeac661e011,CLICK,2010-09-02T20:28:03.314Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
817bf3ce-c78d-4642-af63-def851a76445,93f65b6f-d84c-4c20-80bd-2562205cffb1,dea0f4f1-d835-4a00-a057-111f243bb306,VIEW,1990-08-06T18:43:15.538Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
5020036f-9adb-48f1-b9cd-c59e77002b66,7eb3eae7-1885-448a-8f57-bc7507afb918,f48f854e-5562-44f4-a2f5-b1a93e4f22ba,PURCHASE,2014-10-05T17:01:24.581Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
75d5d68f-c249-4e9d-a74d-a3973df5deff,d7c6b6f3-ba95-43e9-b2a3-b4be4553ec86,619f0a24-e367-472a-9aa6-479c27910c8e,VIEW,2015-03-21T03:48:16.631Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
77c0fc27-063c-47eb-8110-df55d8b08cf3,40d708ea-5c47-4391-9c15-d598283ba354,bb2bec75-5aae-4bd7-ba4e-acb9cf50c4ca,CLICK,1981-02-11T02:39:58.941Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
550d7e31-9523-4c53-afc1-cfaf75d97e42,8dedd09f-e0d1-4435-a3b9-2d679a48f547,e1791238-d374-416b-874c-39fa26056917,CLICK,1978-12-27T01:57:16.788Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
cea81341-4e60-427b-a99c-4a4c6463ca6c,3b465203-2824-45b9-a9df-367e0443df34,f7c484dd-432e-4406-b3cf-d66d80901edf,PURCHASE,2004-12-12T12:57:33.494Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null


event_id,user_id,campaign_id,event_type,event_timestamp,source_file,ingestion_timestamp,rejection_reason
265ede26-f888-4b6e-ae44-4a351a0398b3,null,e709e268-11a4-4286-af8b-04091e4faaa5,CLICK,1987-07-28T08:15:57.296Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00003-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-290-1-c000.csv,2026-02-04T14:40:28.785Z,NULL_USER_ID
046ac710-9adc-4c15-aaa0-dbc412d15a96,null,f9e8ed3f-be1f-4d34-9aed-12f6aea6173f,PURCHASE,2001-11-28T11:36:31.405Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00004-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-291-1-c000.csv,2026-02-04T14:40:28.785Z,NULL_USER_ID
6623a117-4a93-47b5-84e5-5af460f6e548,null,a047b571-6942-4ac0-8861-2b6fdfdb59f1,VIEW,2023-07-13T16:23:36.688Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00005-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-292-1-c000.csv,2026-02-04T14:40:28.785Z,NULL_USER_ID
ec117ed7-ee56-4474-aa38-e233f801a457,null,690c1a53-d4c6-4901-b850-de93a6b623fa,CLICK,1999-06-06T20:36:00.587Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00006-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-287-1-c000.csv,2026-02-04T14:40:28.785Z,NULL_USER_ID
2325d819-6961-40b7-b07a-ee822423409d,null,b87b1d71-d53b-416b-90cc-557a9dedb760,PURCHASE,1982-09-20T12:15:50.616Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00007-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-288-1-c000.csv,2026-02-04T14:40:28.785Z,NULL_USER_ID


#### Escrita na Silver

In [0]:
EVENTS_SILVER_PATH = "/Volumes/main/lakehouse_marketing/silver"

# Resultados válidos
df_valid.write\
        .format("delta")\
        .mode("overwrite")\
        .save(f"{EVENTS_SILVER_PATH}/events")



# Registros rejeitados
df_rejected.write\
        .format("delta")\
        .mode("overwrite")\
        .save(f"{EVENTS_SILVER_PATH}/events_rejected")


#### Validação Dados Silver - Events


In [0]:
df_events_valid = spark.read\
                    .format("delta")\
                    .option("header", "true")\
                    .option("inferSchema", "true")\
                    .load(f"{EVENTS_SILVER_PATH}/events")

display(df_events_valid)
    

event_id,user_id,campaign_id,event_type,event_timestamp,source_file,ingestion_timestamp,rejection_reason
bdd640fb-0667-4ad1-9c80-317fa3b1799d,23b8c1e9-3924-46de-beb1-3b9046685257,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,VIEW,2020-01-18T12:28:35.290Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
0822e8f3-6c03-4199-972a-846916419f82,3b8faa18-37f8-488b-97fc-695a07a0ca6e,8fadc1a6-06cb-4fb3-9a1d-e644815ef6d1,VIEW,1981-02-25T21:46:23.887Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
6b65a6a4-8b81-48f6-b38a-088ca65ed389,47378190-96da-4dac-b2ff-5d2a386ecbe0,c241330b-01a9-471f-9e8a-774bcf36d58b,PURCHASE,2015-03-16T02:48:25.304Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
1ff49b78-8946-4e85-b59c-de66bacfb3d0,142c3fe8-60e7-4113-ac1b-8ca1f91e1d4c,a0ee89ae-d453-4d32-8b0d-bb418d5288f1,PURCHASE,2004-09-11T14:39:00.921Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
3139d32c-93cd-49bf-9c94-1cf0dc98d2c1,a9488d99-0bbb-4599-91ce-5dd2b45ed1f0,fc377a4c-4a15-444d-85e7-ce8a3a578a8e,PURCHASE,1974-06-23T19:50:31.508Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
614ff3d7-19db-4ad0-9dd1-dfb23b982ef8,d58842de-a2bc-472f-b412-b29347294739,5af30553-5ec4-4e08-a9a3-b2e95d65a441,PURCHASE,1981-10-02T12:11:30.860Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
aefcfad8-efc8-4849-b3aa-7efe4458a885,a28defe3-9bf0-4273-9247-6f57a5e5a5ab,3eabedcb-baa8-4dd4-88bd-64072bcfbe01,CLICK,1979-03-02T21:21:43.341Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
ece66fa2-fd51-46e6-851b-4cf36123fdf7,3838b326-8e94-4239-b02b-61c4a3d70628,c4b032cc-d7c5-44a5-9304-317faf42e12f,PURCHASE,2013-07-13T03:50:33.033Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
ce177b4e-0837-48a3-9261-a7ab3aa2e4f9,10f1bc81-448a-4a9e-a6b2-bc5b50c187fc,9132b63e-f162-47e4-a9c3-49e03602f8ac,VIEW,2019-02-28T13:19:24.467Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null
7fcd9eb1-a7ca-4415-b66e-b16f508ebad7,a491f0b2-ea1f-4a65-a27a-984d654821d0,23bed01d-43cf-4fde-a493-3b83757750a9,VIEW,1983-11-02T01:19:11.774Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-1047043620734724595-836d33a1-1461-4511-b273-2bc4ba282a9e-285-1-c000.csv,2026-02-04T14:40:28.785Z,null


In [0]:
df_events_valid.count()

89933